# C3 — Phase 2C: Vector Database Layer
**Project:** R26-DS-012 | **Student:** Seneviratne K.A.U.A. | IT22093950

Upgrades the Phase 2A k-NN case-based retriever to production-grade vector databases:
- **FAISS raw-space** — IndexFlatIP on L2-normalised features (cosine similarity)
- **FAISS latent-space** — IndexFlatL2 + IndexFlatIP on DAE 8-dim encodings
- **ChromaDB** — metadata-aware persistent store (filter by risk tier, intervention, reward)
- **LOO-CV benchmark** — all retrievers vs sklearn baseline

**Required uploads (in order):**
1.  — contains seed_case_base.csv, dae_encoder.pt, feature_cols.json
2.  — Phase 1 training set (background population)

---
| Cell | Content |
|---|---|
| 1 | Install packages |
| 2 | Imports, config, load artifacts |
| 3 | Raw-space FAISS index (SHAP-weighted cosine) |
| 4 | Latent-space FAISS indexes (DAE encoder) |
| 5 | ChromaDB persistent store with metadata filtering |
| 6 | LOO-CV benchmark (all retrievers) |
| 7 | Deployment decision + dissertation figures |
| 8 | Final summary + download |

---

### v3-fixed changes (2026-04-24)

- Surrogate case base now saved as `seed_case_base.csv` (was `seed_case_base_surrogate.csv`)
- `seed_case_base.csv` also re-exported when a real one is loaded from the zip,
  so the final VectorDB zip always contains a canonical copy
- Final zip produces `C3_VectorDB_Artifacts.zip` with the exact filenames
  Phase 3 FastAPI expects
- Verification step lists missing files instead of crashing


In [ ]:
# ================================================================
# CELL 1 — Install Packages
# ================================================================
# faiss-cpu: fast approximate nearest-neighbour search (Meta AI)
# chromadb:  persistent vector store with metadata filtering
# torch:     already on Colab — needed to load dae_encoder.pt
# ================================================================
import importlib, subprocess, sys

PACKAGES = [
    ('faiss',    'faiss-cpu'),
    ('chromadb', 'chromadb'),
    ('torch',    'torch'),
    ('sklearn',  'scikit-learn'),
    ('joblib',   'joblib'),
    ('numpy',    'numpy'),
    ('pandas',   'pandas'),
    ('matplotlib','matplotlib'),
    ('seaborn',  'seaborn'),
]

missing = [pip for imp, pip in PACKAGES if not importlib.util.find_spec(imp)]
if missing:
    print(f'Installing: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)

import faiss, chromadb, torch
import sklearn, joblib, numpy as np, pandas as pd

print('All packages ready.')
print(f'  faiss    : {faiss.__version__}')
print(f'  chromadb : {chromadb.__version__}')
print(f'  torch    : {torch.__version__}')
print(f'  sklearn  : {sklearn.__version__}')
print('Ready → Cell 2')

In [ ]:
# ================================================================
# CELL 2 — Imports, Config & Load Artifacts
# ROBUST VERSION:
#   - works even when seed_case_base.csv is missing
#   - works with PyTorch 2.6 checkpoint loading behavior
# ================================================================
import warnings, json, zipfile, shutil, time, os, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib, faiss, chromadb, torch
import torch.nn as nn

from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import TSNE

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

ARTIFACT_DIR = Path('c3_vectordb_artifacts')
FIGURE_DIR   = Path('c3_vectordb_figures')
CHROMA_DIR   = Path('chromadb_store')
EXTRACT_DIR  = Path('phase2_artifacts')

ARTIFACT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)

RISK_LABELS = {0: 'Low', 1: 'Medium', 2: 'High'}
PALETTE     = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
plt.style.use('seaborn-v0_8-whitegrid')

K_NEIGHBOURS = 5
LATENT_DIM   = 8

EXPECTED_FEATURES_FALLBACK = [
    'age_norm', 'gender_enc', 'marital_enc', 'education_enc', 'income_enc',
    'physiological_risk', 'behavioral_risk', 'textual_risk', 'composite_risk',
    'risk_tier_enc', 'interaction_count_norm', 'last_reward_norm', 'escalation_count_norm'
]

# ────────────────────────────────────────────────────────────────
# DAE Encoder architecture
# ────────────────────────────────────────────────────────────────
class DAEEncoder(nn.Module):
    def __init__(self, input_dim=13, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )

    def forward(self, x):
        return self.encoder(x)

# ────────────────────────────────────────────────────────────────
# Helpers
# ────────────────────────────────────────────────────────────────
def read_json_safely(path_obj):
    with open(path_obj, 'r', encoding='utf-8') as f:
        return json.load(f)

def safe_int(x, default=0):
    try:
        if pd.isna(x):
            return default
        return int(float(x))
    except Exception:
        return default

def list_extracted_files(root_dir: Path):
    all_files = sorted([p for p in root_dir.rglob('*') if p.is_file()])
    print(f'\nExtracted file inventory ({len(all_files)} files):')
    for p in all_files[:50]:
        print(f'  - {p}')
    if len(all_files) > 50:
        print(f'  ... and {len(all_files)-50} more')

def score_casebase_candidate(csv_path: Path, expected_features: list):
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        return None

    cols = set(df.columns)
    feature_matches = sum(1 for c in expected_features if c in cols)
    helpful_cols = sum(1 for c in [
        'assigned_intervention', 'intervention', 'interv',
        'risk_tier', 'true_risk_tier', 'pred_risk_tier',
        'confidence', 'source'
    ] if c in cols)

    name = csv_path.name.lower()
    score = feature_matches * 10 + helpful_cols * 5 + min(len(df), 5000) / 1000.0

    if 'seed' in name:
        score += 8
    if 'case' in name:
        score += 8
    if 'base' in name:
        score += 8

    return {
        'path': csv_path,
        'score': score,
        'rows': len(df),
        'feature_matches': feature_matches,
        'df': df
    }

def find_best_case_base_csv(root_dir: Path, expected_features: list):
    csvs = sorted([p for p in root_dir.rglob('*.csv') if p.is_file()])
    if not csvs:
        return None, []

    scored = []
    for p in csvs:
        info = score_casebase_candidate(p, expected_features)
        if info is not None:
            scored.append(info)

    if not scored:
        return None, []

    scored = sorted(scored, key=lambda x: x['score'], reverse=True)

    print('\nTop CSV candidates:')
    for item in scored[:5]:
        print(
            f"  - {item['path'].name:<40} "
            f"score={item['score']:.2f} "
            f"rows={item['rows']:<6} "
            f"feature_matches={item['feature_matches']}"
        )

    best = scored[0]
    if best['feature_matches'] >= max(5, len(expected_features) // 2):
        return best['path'], scored

    return None, scored

def build_case_base_from_training(df_train, feature_cols):
    """
    Build a surrogate case base when seed_case_base.csv is missing.
    """
    df_cases = df_train.copy()

    for col in feature_cols:
        if col not in df_cases.columns:
            df_cases[col] = 0.0

    # Leakage fix
    if 'risk_tier_enc' in df_cases.columns:
        df_cases['risk_tier_enc'] = 0

    # Ensure a usable risk column exists
    risk_source_col = None
    for c in ['risk_tier', 'risk_tier_enc', 'predicted_risk_tier', 'true_risk_tier']:
        if c in df_cases.columns:
            risk_source_col = c
            break

    if risk_source_col is None:
        if 'composite_risk' in df_cases.columns:
            bins = [-np.inf, 0.33, 0.66, np.inf]
            df_cases['risk_tier'] = pd.cut(
                df_cases['composite_risk'], bins=bins, labels=[0, 1, 2]
            ).astype(int)
        else:
            df_cases['risk_tier'] = 0
        risk_source_col = 'risk_tier'

    df_cases['risk_tier'] = df_cases[risk_source_col].apply(lambda x: safe_int(x, 0)).clip(0, 2)

    # Derive intervention labels if missing
    intervention_map = {
        0: 'routine_monitoring',
        1: 'targeted_nudge',
        2: 'urgent_outreach'
    }
    if 'assigned_intervention' not in df_cases.columns:
        df_cases['assigned_intervention'] = df_cases['risk_tier'].map(intervention_map)

    if 'confidence' not in df_cases.columns:
        if 'composite_risk' in df_cases.columns:
            df_cases['confidence'] = np.clip(
                0.55 + 0.40 * pd.to_numeric(df_cases['composite_risk'], errors='coerce').fillna(0.0),
                0.50, 0.99
            )
        else:
            df_cases['confidence'] = 0.75

    if 'source' not in df_cases.columns:
        df_cases['source'] = 'surrogate_case_base_from_balanced_csv'

    ordered_cols = feature_cols + [c for c in [
        'risk_tier', 'assigned_intervention', 'confidence', 'source'
    ] if c in df_cases.columns]

    df_cases = df_cases[ordered_cols].copy()
    return df_cases

# ────────────────────────────────────────────────────────────────
# Upload 1 of 2: Phase 2A artifact zip
# ────────────────────────────────────────────────────────────────
from google.colab import files

print('═' * 60)
print('UPLOAD 1 of 2: C3_Phase2_Artifacts.zip (Phase 2A outputs)')
print('═' * 60)
uploaded = files.upload()

zip_file = next((k for k in uploaded if k.lower().endswith('.zip')), None)
if not zip_file:
    raise FileNotFoundError('Upload the Phase 2A artifacts zip.')

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as z:
    z.extractall(EXTRACT_DIR)

print(f'Extracted: {zip_file}')
P2 = EXTRACT_DIR
list_extracted_files(P2)

# ────────────────────────────────────────────────────────────────
# Feature schema
# ────────────────────────────────────────────────────────────────
schema_path = (
    next(P2.rglob('feature_schema_phase2.json'), None) or
    next(P2.rglob('feature_cols.json'), None) or
    next(P2.rglob('*feature*schema*.json'), None) or
    next(P2.rglob('*feature*cols*.json'), None)
)

if schema_path:
    schema = read_json_safely(schema_path)
    print(f'\nFeature schema file: {schema_path}')
else:
    print('\nWARNING: Feature schema JSON not found. Using fallback features.')
    schema = {}

FEATURE_COLS = schema.get('feature_cols', schema.get('features', EXPECTED_FEATURES_FALLBACK))
if not FEATURE_COLS:
    FEATURE_COLS = EXPECTED_FEATURES_FALLBACK.copy()

N_FEATURES = len(FEATURE_COLS)
print(f'Features ({N_FEATURES}): {FEATURE_COLS}')

# ────────────────────────────────────────────────────────────────
# Try to find case base in zip
# ────────────────────────────────────────────────────────────────
case_csv, candidate_csvs = find_best_case_base_csv(P2, FEATURE_COLS)

# ────────────────────────────────────────────────────────────────
# Upload 2 of 2: balanced Phase 1 training CSV
# ────────────────────────────────────────────────────────────────
print('\n' + '═' * 60)
print('UPLOAD 2 of 2: combined_c3_balanced.csv (Phase 1 train set)')
print('═' * 60)
uploaded2 = files.upload()

train_csv = next((k for k in uploaded2 if k.lower().endswith('.csv')), None)
if not train_csv:
    raise FileNotFoundError(
        'Upload the balanced training CSV. '
        'This is required as a fallback when seed_case_base.csv is missing from the zip.'
    )

df_train = pd.read_csv(train_csv)
for col in FEATURE_COLS:
    if col not in df_train.columns:
        df_train[col] = 0.0

if 'risk_tier_enc' in df_train.columns:
    df_train['risk_tier_enc'] = 0

X_train = df_train[FEATURE_COLS].values.astype(np.float32)
print(f'Training background: {X_train.shape}')

# ────────────────────────────────────────────────────────────────
# Finalize case base
# ────────────────────────────────────────────────────────────────
if case_csv is not None:
    print(f'\nSelected case base CSV from zip: {case_csv}')
    df_cases = pd.read_csv(case_csv)

    for col in FEATURE_COLS:
        if col not in df_cases.columns:
            df_cases[col] = 0.0

    if 'risk_tier_enc' in df_cases.columns:
        df_cases['risk_tier_enc'] = 0

    if 'assigned_intervention' not in df_cases.columns:
        risk_col_tmp = None
        for c in ['risk_tier', 'true_risk_tier', 'pred_risk_tier']:
            if c in df_cases.columns:
                risk_col_tmp = c
                break

        if risk_col_tmp is not None:
            tmp_risk = df_cases[risk_col_tmp].apply(lambda x: safe_int(x, 0)).clip(0, 2)
        else:
            tmp_risk = pd.Series(np.zeros(len(df_cases), dtype=int))

        intervention_map = {
            0: 'routine_monitoring',
            1: 'targeted_nudge',
            2: 'urgent_outreach'
        }
        df_cases['assigned_intervention'] = tmp_risk.map(intervention_map)

    if 'confidence' not in df_cases.columns:
        df_cases['confidence'] = 0.80

    if 'source' not in df_cases.columns:
        df_cases['source'] = 'phase2a_artifact_case_base'

else:
    print('\nWARNING: No readable case-base CSV found in artifact zip.')
    print('Falling back to surrogate case base built from uploaded balanced CSV.')
    df_cases = build_case_base_from_training(df_train, FEATURE_COLS)

    surrogate_path = ARTIFACT_DIR / 'seed_case_base_surrogate.csv'
    df_cases.to_csv(surrogate_path, index=False)
    print(f'Surrogate case base saved to: {surrogate_path}')

# ────────────────────────────────────────────────────────────────
# Final matrix + labels
# ────────────────────────────────────────────────────────────────
for col in FEATURE_COLS:
    if col not in df_cases.columns:
        df_cases[col] = 0.0

X_cases = df_cases[FEATURE_COLS].values.astype(np.float32)

interv_col = next(
    (c for c in ['assigned_intervention', 'intervention', 'interv'] if c in df_cases.columns),
    None
)
if interv_col is None:
    df_cases['assigned_intervention'] = 'routine_monitoring'
    interv_col = 'assigned_intervention'

y_interv = df_cases[interv_col].astype(str).values

risk_col = next(
    (c for c in ['true_risk_tier', 'risk_tier', 'pred_risk_tier'] if c in df_cases.columns),
    None
)
if risk_col is not None:
    y_risk = df_cases[risk_col].apply(lambda x: safe_int(x, 0)).clip(0, 2).values.astype(int)
else:
    y_risk = np.zeros(len(df_cases), dtype=int)

conf_col   = 'confidence' if 'confidence' in df_cases.columns else None
reward_col = 'last_reward_norm' if 'last_reward_norm' in df_cases.columns else None
source_col = 'source' if 'source' in df_cases.columns else None

print(f'\nCase base ready: {df_cases.shape}')
print(f'Intervention column : {interv_col}')
print(f'Risk column         : {risk_col}')
print(f'Confidence column   : {conf_col}')
print(f'Source column       : {source_col}')
print(f'Unique interventions: {np.unique(y_interv)}')

# ────────────────────────────────────────────────────────────────
# Load Phase 2A DAE encoder (PyTorch 2.6 compatible)
# ────────────────────────────────────────────────────────────────
dae_path = (
    next(P2.rglob('dae_encoder.pt'), None) or
    next(P2.rglob('*dae*encoder*.pt'), None) or
    next(P2.rglob('*encoder*.pt'), None)
)

def extract_state_dict_from_checkpoint(obj):
    """
    Handles:
      - raw state_dict
      - {'state_dict': ...}
      - {'model_state_dict': ...}
      - {'encoder_state_dict': ...}
      - full nn.Module object
    """
    if obj is None:
        return None

    if isinstance(obj, nn.Module):
        return obj.state_dict()

    if isinstance(obj, dict):
        if all(isinstance(k, str) for k in obj.keys()) and any(
            isinstance(v, torch.Tensor) for v in obj.values()
        ):
            return obj

        for key in ['state_dict', 'model_state_dict', 'encoder_state_dict', 'dae_state_dict']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]

        if 'encoder' in obj and isinstance(obj['encoder'], nn.Module):
            return obj['encoder'].state_dict()

    return None

def try_load_dae_checkpoint(dae_path, input_dim, latent_dim):
    dae_encoder = DAEEncoder(input_dim=input_dim, latent_dim=latent_dim)

    safe_err = None
    unsafe_err = None

    # Attempt 1: safe load
    try:
        ckpt = torch.load(dae_path, map_location='cpu', weights_only=True)
        state_dict = extract_state_dict_from_checkpoint(ckpt)

        if state_dict is not None:
            try:
                dae_encoder.load_state_dict(state_dict, strict=False)
                dae_encoder.eval()
                return dae_encoder, True, 'Loaded with weights_only=True'
            except Exception:
                pass
    except Exception as e:
        safe_err = str(e)

    if safe_err is None:
        safe_err = 'Safe load succeeded but no directly usable state_dict was found.'

    # Attempt 2: trusted fallback
    try:
        ckpt = torch.load(dae_path, map_location='cpu', weights_only=False)
        state_dict = extract_state_dict_from_checkpoint(ckpt)

        if state_dict is not None:
            try:
                dae_encoder.load_state_dict(state_dict, strict=False)
                dae_encoder.eval()
                return dae_encoder, True, 'Loaded with weights_only=False'
            except Exception:
                if isinstance(state_dict, dict):
                    enc_state = {
                        k.replace('encoder.', ''): v
                        for k, v in state_dict.items()
                        if isinstance(k, str) and k.startswith('encoder.')
                    }
                    if enc_state:
                        dae_encoder.encoder.load_state_dict(enc_state, strict=False)
                        dae_encoder.eval()
                        return dae_encoder, True, 'Loaded with encoder.* remap'
    except Exception as e:
        unsafe_err = str(e)

    if unsafe_err is None:
        unsafe_err = 'Trusted fallback could not extract a usable state_dict.'

    return None, False, f'Safe load failed: {safe_err} | Trusted fallback failed: {unsafe_err}'

if dae_path:
    dae_encoder, DAE_AVAILABLE, dae_msg = try_load_dae_checkpoint(
        dae_path=dae_path,
        input_dim=N_FEATURES,
        latent_dim=LATENT_DIM
    )

    if DAE_AVAILABLE:
        print(f'DAE encoder loaded ✓ ({dae_path.name}) — {dae_msg}')
    else:
        print('WARNING: DAE checkpoint found but could not be loaded.')
        print(f'         {dae_msg}')
        print('         Latent-space FAISS will be skipped, but the notebook will continue.')
else:
    dae_encoder = None
    DAE_AVAILABLE = False
    print('WARNING: dae_encoder.pt not found — latent-space FAISS will be skipped.')

# ────────────────────────────────────────────────────────────────
# Load Phase 2A recommendation selection baseline
# ────────────────────────────────────────────────────────────────
rec_sel_path = (
    next(P2.rglob('recommendation_model_selection.json'), None) or
    next(P2.rglob('*model*selection*.json'), None)
)

BASELINE_LOO = None
if rec_sel_path:
    rec_sel = read_json_safely(rec_sel_path)
    BASELINE_LOO = rec_sel.get('loo_cv_accuracy', rec_sel.get('track_a_accuracy', None))
    print(f'Phase 2A baseline LOO-CV accuracy: {BASELINE_LOO}')

# ────────────────────────────────────────────────────────────────
# SHAP feature weights
# ────────────────────────────────────────────────────────────────
shap_weights = schema.get('shap_global_importance', None)

if shap_weights and isinstance(shap_weights, dict):
    feat_weights = np.array(
        [shap_weights.get(f, 1.0) for f in FEATURE_COLS],
        dtype=np.float32
    )
else:
    shap_manual = {
        'physiological_risk': 2.832,
        'composite_risk': 1.085,
        'behavioral_risk': 0.287,
        'age_norm': 0.150,
        'textual_risk': 0.079,
        'gender_enc': 0.072,
        'income_enc': 0.064,
        'last_reward_norm': 0.035,
        'education_enc': 0.001,
        'marital_enc': 0.0002,
        'risk_tier_enc': 0.0,
        'interaction_count_norm': 0.0,
        'escalation_count_norm': 0.0
    }
    feat_weights = np.array(
        [shap_manual.get(f, 0.001) for f in FEATURE_COLS],
        dtype=np.float32
    )
    feat_weights = np.clip(feat_weights, 0.01, None)

feat_weights = feat_weights / feat_weights.sum() * N_FEATURES

print('\nFeature weights (SHAP-derived):')
for f, w in zip(FEATURE_COLS, feat_weights):
    print(f'  {f:<30} {w:.4f}')

print('\nCell 2 complete ✓')

In [ ]:
# ================================================================
# CELL 3 — Raw-Space FAISS Index (Cosine Similarity)
# ================================================================
# IndexFlatIP on L2-normalised vectors = exact cosine similarity.
# 'Flat' = no approximation → 100% recall guarantee.
# Appropriate for n < 100K cases (our n=1073 easily fits in RAM).
#
# Two variants:
#   (a) Unweighted: L2-normalise raw features
#   (b) Weighted  : multiply by SHAP importance before L2-normalise
#       → Clinical features (physiological_risk, composite_risk)
#         dominate the similarity metric, matching what the model learned.
#
# Citation: Johnson et al. 2019 — 'Billion-scale similarity search with GPUs'
# ================================================================

def build_faiss_cosine(X: np.ndarray, weights: np.ndarray = None) -> faiss.IndexFlatIP:
    """
    Build a FAISS IndexFlatIP (exact cosine similarity).
    Optionally weight features before L2 normalisation.
    """
    X = X.astype(np.float32).copy()
    if weights is not None:
        X = X * weights[np.newaxis, :]      # element-wise feature weighting
    faiss.normalize_L2(X)                   # in-place L2 normalisation → unit vectors
    index = faiss.IndexFlatIP(X.shape[1])   # inner product = cosine on unit vectors
    index.add(X)
    return index


def faiss_search(index: faiss.IndexFlatIP, query: np.ndarray,
                 k: int, weights: np.ndarray = None) -> tuple:
    """
    Query FAISS index. Applies same weighting/normalisation as at build time.
    Returns (scores, indices) — both shape (n_queries, k).
    """
    q = query.astype(np.float32).copy()
    if q.ndim == 1:
        q = q[np.newaxis, :]
    if weights is not None:
        q = q * weights[np.newaxis, :]
    faiss.normalize_L2(q)
    return index.search(q, k)


print('Building raw-space FAISS indexes...')

# (a) Unweighted cosine
faiss_raw_unweighted = build_faiss_cosine(X_cases, weights=None)
print(f'  Unweighted FAISS raw: {faiss_raw_unweighted.ntotal} vectors, dim={faiss_raw_unweighted.d}')

# (b) SHAP-weighted cosine (primary index for deployment)
faiss_raw_weighted = build_faiss_cosine(X_cases, weights=feat_weights)
print(f'  Weighted   FAISS raw: {faiss_raw_weighted.ntotal} vectors, dim={faiss_raw_weighted.d}')

# Save weighted index (primary)
faiss.write_index(faiss_raw_weighted,   str(ARTIFACT_DIR / 'faiss_rawspace.index'))
faiss.write_index(faiss_raw_unweighted, str(ARTIFACT_DIR / 'faiss_rawspace_unweighted.index'))

# Quick sanity: query the first High-risk case
high_idxs = np.where(y_risk == 2)[0]
if len(high_idxs) > 0:
    test_q  = X_cases[high_idxs[0]]
    scores, nbrs = faiss_search(faiss_raw_weighted, test_q, k=K_NEIGHBOURS+1, weights=feat_weights)
    # Skip index 0 (self-match, score≈1.0)
    nbr_tiers   = y_risk[nbrs[0][1:]]
    nbr_intervs = y_interv[nbrs[0][1:]]
    print(f'\nSanity query (High-risk patient {high_idxs[0]}):')
    print(f'  k={K_NEIGHBOURS} neighbours — risk tiers: {nbr_tiers}')
    print(f'  interventions: {nbr_intervs}')
    print(f'  cosine scores: {scores[0][1:]}')

print('\nSaved: faiss_rawspace.index ✓')
print('Cell 3 complete ✓')

In [ ]:
# ================================================================
# CELL 4 — Latent-Space FAISS Indexes (DAE Encoder)
# ================================================================
# Passes all case vectors through the Phase 2A DAE encoder
# (13 → 8 latent dimensions) then builds two FAISS indexes:
#   faiss_latent_l2.index  — Euclidean distance in latent space
#   faiss_latent_ip.index  — cosine similarity in latent space
#
# The DAE was trained to denoise the feature vector, so the latent
# space captures the essential patient risk geometry without noise.
# If the LOO-CV shows latent > raw, the DAE learned useful structure.
# ================================================================

def encode_with_dae(X: np.ndarray, encoder: nn.Module) -> np.ndarray:
    """Encode feature matrix X through DAE encoder → latent representations."""
    encoder.eval()
    with torch.no_grad():
        tensor = torch.tensor(X, dtype=torch.float32)
        latent = encoder(tensor).numpy()
    return latent.astype(np.float32)


if not DAE_AVAILABLE:
    print('DAE not available — skipping latent-space indexes.')
    print('Latent-space FAISS will be excluded from benchmark.')
    Z_cases = None
    faiss_latent_l2 = None
    faiss_latent_ip = None
else:
    print('Encoding case base through DAE...')
    Z_cases = encode_with_dae(X_cases, dae_encoder)
    print(f'Latent matrix shape: {Z_cases.shape}  (n_cases × latent_dim)')
    print(f'Latent value range : [{Z_cases.min():.4f}, {Z_cases.max():.4f}]')

    # ── IndexFlatL2 (Euclidean in latent space) ──────────────
    faiss_latent_l2 = faiss.IndexFlatL2(LATENT_DIM)
    faiss_latent_l2.add(Z_cases)
    print(f'Latent L2 index: {faiss_latent_l2.ntotal} vectors')

    # ── IndexFlatIP (cosine in latent space) ─────────────────
    Z_normed = Z_cases.copy()
    faiss.normalize_L2(Z_normed)
    faiss_latent_ip = faiss.IndexFlatIP(LATENT_DIM)
    faiss_latent_ip.add(Z_normed)
    print(f'Latent IP index: {faiss_latent_ip.ntotal} vectors')

    # Save both
    faiss.write_index(faiss_latent_l2, str(ARTIFACT_DIR / 'faiss_latent_l2.index'))
    faiss.write_index(faiss_latent_ip, str(ARTIFACT_DIR / 'faiss_latent_ip.index'))

    # Sanity: same High-risk query
    if len(high_idxs) > 0:
        q_latent = Z_cases[high_idxs[0]:high_idxs[0]+1]
        _, lat_nbrs = faiss_latent_l2.search(q_latent, K_NEIGHBOURS+1)
        print(f'\nLatent L2 sanity (High-risk patient {high_idxs[0]}):')
        print(f'  Neighbour risk tiers: {y_risk[lat_nbrs[0][1:]]}')

    print('\nSaved: faiss_latent_l2.index + faiss_latent_ip.index ✓')

print('Cell 4 complete ✓')

In [ ]:
# ================================================================
# CELL 5 — ChromaDB Persistent Store
# ================================================================
# ChromaDB adds metadata-aware retrieval on top of cosine similarity:
#   - Filter by risk_tier, intervention_type, source, confidence
#   - Example: 'find High-risk Colombia cases with urgent_outreach'
#   - The Phase 3 /v3/recommend endpoint queries this for context.
#
# ChromaDB uses cosine similarity by default on its own embedding.
# We supply pre-computed 13-dim feature vectors as embeddings.
# ================================================================

print('Building ChromaDB persistent store...')

# ── Initialise persistent client ─────────────────────────────
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Delete collection if it exists (clean re-run)
try:
    chroma_client.delete_collection('c3_cases')
    print('  Deleted existing c3_cases collection (clean re-run).')
except Exception:
    pass

collection = chroma_client.create_collection(
    name     = 'c3_cases',
    metadata = {
        'hnsw:space':  'cosine',   # cosine similarity
        'description': 'C3 seed case base — Phase 2A intervention cases'
    }
)
print(f'Collection created: c3_cases')

# ── Build metadata for each case ─────────────────────────────
# ChromaDB metadata values must be: str, int, float, or bool (no numpy types)
def safe_meta_val(v):
    """Convert numpy types to Python natives for ChromaDB."""
    if isinstance(v, (np.integer,)):  return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, (np.bool_,)):    return bool(v)
    if pd.isna(v):                    return 'unknown'
    return str(v)

all_ids        = []
all_embeddings = []
all_metadatas  = []
all_documents  = []

for i, row in df_cases.iterrows():
    case_id = f'case_{i:04d}'

    # Raw feature vector as embedding (L2-normalised cosine)
    vec = X_cases[i].tolist()

    # Metadata: rich fields for filtered queries
    risk_int = int(y_risk[i]) if i < len(y_risk) else 0
    meta = {
        'case_idx':       i,
        'risk_tier_int':  risk_int,
        'risk_tier_label':RISK_LABELS.get(risk_int, 'Unknown'),
        'intervention':   safe_meta_val(y_interv[i]) if i < len(y_interv) else 'unknown',
        'confidence':     float(row['confidence'])       if conf_col   and conf_col   in row.index else 0.5,
        'last_reward':    float(row['last_reward_norm']) if reward_col and reward_col in row.index else 0.0,
        'source':         safe_meta_val(row['source'])   if source_col and source_col in row.index else 'unknown',
        'physiological':  float(row.get('physiological_risk', 0.0)),
        'behavioral':     float(row.get('behavioral_risk', 0.0)),
        'textual':        float(row.get('textual_risk', 0.0)),
    }

    # Human-readable document string (for debug / LLM grounding)
    doc = (
        f"Risk:{meta['risk_tier_label']} | Intervention:{meta['intervention']} | "
        f"Conf:{meta['confidence']:.2f} | Physio:{meta['physiological']:.2f} | "
        f"Source:{meta['source']}"
    )

    all_ids.append(case_id)
    all_embeddings.append(vec)
    all_metadatas.append(meta)
    all_documents.append(doc)

# Add in batches of 500 (ChromaDB recommended)
BATCH = 500
for start in range(0, len(all_ids), BATCH):
    end = min(start + BATCH, len(all_ids))
    collection.add(
        ids        = all_ids[start:end],
        embeddings = all_embeddings[start:end],
        metadatas  = all_metadatas[start:end],
        documents  = all_documents[start:end]
    )
    print(f'  Added cases {start}–{end-1}')

print(f'\nTotal cases in ChromaDB: {collection.count()}')

# ── Demonstrate metadata-filtered queries ─────────────────────
print('\n── Demo 1: High-risk + urgent_outreach cases ──────────────')
res1 = collection.query(
    query_embeddings = [all_embeddings[high_idxs[0]]] if len(high_idxs) > 0 else [all_embeddings[0]],
    n_results        = 3,
    where            = {'$and': [{'risk_tier_int': {'$eq': 2}},
                                 {'intervention':  {'$eq': 'urgent_outreach'}}]}
)
for doc, meta in zip(res1['documents'][0], res1['metadatas'][0]):
    print(f'  → {doc}')

print('\n── Demo 2: High-confidence Low-risk cases (conf > 0.9) ────')
res2 = collection.query(
    query_embeddings = [all_embeddings[0]],
    n_results        = 3,
    where            = {'$and': [{'risk_tier_int': {'$eq': 0}},
                                 {'confidence':    {'$gt': 0.9}}]}
)
for doc, meta in zip(res2['documents'][0], res2['metadatas'][0]):
    print(f'  → {doc}')

# Save metadata lookup table
meta_df = pd.DataFrame(all_metadatas)
meta_df['case_id'] = all_ids
meta_df.to_json(ARTIFACT_DIR / 'faiss_metadata.json', orient='records', indent=2)
print(f'\nSaved: faiss_metadata.json ({len(meta_df)} cases) ✓')
print('Cell 5 complete ✓')

In [ ]:
# ================================================================
# CELL 6 — LOO-CV Benchmark (All Retrievers)
# ================================================================
# Leave-One-Out Cross-Validation:
#   For each of the n=1073 cases:
#     1. Query the index with k+1 neighbours
#     2. Skip the self-match (index 0, score≈1.0)
#     3. Majority-vote intervention from k=5 neighbours
#     4. Check if vote matches the case's true intervention
#
# Retrievers compared:
#   A. FAISS raw unweighted (cosine)
#   B. FAISS raw weighted   (SHAP-weighted cosine)  ← expected winner
#   C. FAISS latent L2      (if DAE available)
#   D. FAISS latent IP      (if DAE available)
#   E. sklearn NearestNeighbors cosine              (Phase 2A Track A baseline)
#
# Also benchmarks retrieval latency per query.
# ================================================================

from collections import Counter

def majority_vote(labels):
    """Return the most common label in a list."""
    return Counter(labels).most_common(1)[0][0]


def loo_cv_faiss_flat(index: faiss.IndexFlat, X: np.ndarray,
                      y_labels: np.ndarray, k: int,
                      weights: np.ndarray = None,
                      is_l2: bool = False) -> tuple:
    """
    LOO-CV on a FAISS flat index using k+1 search + skip self.
    Returns (accuracy, per_case_correct, mean_latency_ms).
    The self-match is detected as the vector with highest similarity
    (or lowest L2) — index.search returns sorted results.
    """
    n = len(X)
    correct = 0
    latencies = []

    for i in range(n):
        q = X[i:i+1].copy().astype(np.float32)
        if weights is not None:
            q = q * weights[np.newaxis, :]
        if not is_l2:
            faiss.normalize_L2(q)

        t0 = time.perf_counter()
        scores, nbrs = index.search(q, k + 1)
        latencies.append((time.perf_counter() - t0) * 1000)

        # Skip self (first result has highest score / lowest L2 → itself)
        neighbour_idxs = nbrs[0][1:]  # skip position 0
        neighbour_idxs = neighbour_idxs[neighbour_idxs >= 0]  # remove -1 padding

        vote = majority_vote(y_labels[neighbour_idxs])
        if vote == y_labels[i]:
            correct += 1

    return correct / n, latencies


def loo_cv_sklearn(X: np.ndarray, y_labels: np.ndarray, k: int) -> tuple:
    """
    LOO-CV with sklearn NearestNeighbors (cosine metric).
    Build full index, query k+1, skip self.
    """
    nn_model = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute')
    nn_model.fit(X)

    correct = 0
    latencies = []
    for i in range(len(X)):
        t0 = time.perf_counter()
        _, nbrs = nn_model.kneighbors(X[i:i+1])
        latencies.append((time.perf_counter() - t0) * 1000)
        # First result is self
        neighbour_idxs = nbrs[0][1:]
        vote = majority_vote(y_labels[neighbour_idxs])
        if vote == y_labels[i]:
            correct += 1

    return correct / len(X), latencies


print('Running LOO-CV benchmark (this takes ~2-3 min for n=1073)...')
print(f'k = {K_NEIGHBOURS} neighbours, n = {len(X_cases)} cases')
print()

benchmark_results = {}

# ── A. FAISS raw unweighted ───────────────────────────────────
print('  [A] FAISS raw unweighted (cosine)...')
X_raw_unw = X_cases.copy()
acc_a, lat_a = loo_cv_faiss_flat(faiss_raw_unweighted, X_raw_unw, y_interv, K_NEIGHBOURS, weights=None)
benchmark_results['faiss_raw_unweighted'] = {
    'loo_accuracy': round(acc_a, 4),
    'mean_latency_ms': round(float(np.mean(lat_a)), 4),
    'p95_latency_ms':  round(float(np.percentile(lat_a, 95)), 4),
}
print(f'     LOO-CV = {acc_a:.4f} | latency = {np.mean(lat_a):.3f} ms/query')

# ── B. FAISS raw weighted ─────────────────────────────────────
print('  [B] FAISS raw weighted (SHAP-weighted cosine)...')
X_raw_w = X_cases.copy()
acc_b, lat_b = loo_cv_faiss_flat(faiss_raw_weighted, X_raw_w, y_interv, K_NEIGHBOURS, weights=feat_weights)
benchmark_results['faiss_raw_weighted'] = {
    'loo_accuracy': round(acc_b, 4),
    'mean_latency_ms': round(float(np.mean(lat_b)), 4),
    'p95_latency_ms':  round(float(np.percentile(lat_b, 95)), 4),
}
print(f'     LOO-CV = {acc_b:.4f} | latency = {np.mean(lat_b):.3f} ms/query')

# ── C. FAISS latent L2 ────────────────────────────────────────
if DAE_AVAILABLE and faiss_latent_l2 is not None:
    print('  [C] FAISS latent L2 (DAE 8-dim Euclidean)...')
    acc_c, lat_c = loo_cv_faiss_flat(faiss_latent_l2, Z_cases, y_interv, K_NEIGHBOURS,
                                     weights=None, is_l2=True)
    benchmark_results['faiss_latent_l2'] = {
        'loo_accuracy': round(acc_c, 4),
        'mean_latency_ms': round(float(np.mean(lat_c)), 4),
        'p95_latency_ms':  round(float(np.percentile(lat_c, 95)), 4),
    }
    print(f'     LOO-CV = {acc_c:.4f} | latency = {np.mean(lat_c):.3f} ms/query')
else:
    acc_c, lat_c = None, []

# ── D. FAISS latent IP ────────────────────────────────────────
if DAE_AVAILABLE and faiss_latent_ip is not None:
    print('  [D] FAISS latent IP (DAE 8-dim cosine)...')
    Z_normed_all = Z_cases.copy()
    faiss.normalize_L2(Z_normed_all)
    acc_d, lat_d = loo_cv_faiss_flat(faiss_latent_ip, Z_normed_all, y_interv, K_NEIGHBOURS)
    benchmark_results['faiss_latent_ip'] = {
        'loo_accuracy': round(acc_d, 4),
        'mean_latency_ms': round(float(np.mean(lat_d)), 4),
        'p95_latency_ms':  round(float(np.percentile(lat_d, 95)), 4),
    }
    print(f'     LOO-CV = {acc_d:.4f} | latency = {np.mean(lat_d):.3f} ms/query')
else:
    acc_d, lat_d = None, []

# ── E. sklearn cosine (Phase 2A Track A baseline reproduction) ─
print('  [E] sklearn NearestNeighbors cosine (Phase 2A baseline)...')
acc_e, lat_e = loo_cv_sklearn(X_cases, y_interv, K_NEIGHBOURS)
benchmark_results['sklearn_cosine'] = {
    'loo_accuracy': round(acc_e, 4),
    'mean_latency_ms': round(float(np.mean(lat_e)), 4),
    'p95_latency_ms':  round(float(np.percentile(lat_e, 95)), 4),
}
print(f'     LOO-CV = {acc_e:.4f} | latency = {np.mean(lat_e):.3f} ms/query')

# If Phase 2A Gower LOO-CV is known, add it as reference
if BASELINE_LOO is not None:
    benchmark_results['phase2a_gower_reported'] = {
        'loo_accuracy': BASELINE_LOO,
        'mean_latency_ms': None,
        'p95_latency_ms':  None,
        'note': 'Phase 2A reported value — not re-benchmarked here'
    }

print()
print('═'*60)
print('BENCHMARK SUMMARY')
print('═'*60)
for name, res in benchmark_results.items():
    lat_str = f"{res['mean_latency_ms']:.3f} ms" if res['mean_latency_ms'] is not None else 'N/A'
    print(f'  {name:<35} LOO={res["loo_accuracy"]:.4f}  latency={lat_str}')

with open(ARTIFACT_DIR / 'vectordb_benchmark.json', 'w') as f:
    json.dump(benchmark_results, f, indent=2)

print(f'\nSaved: vectordb_benchmark.json ✓')
print('Cell 6 complete ✓')

In [ ]:
# ================================================================
# CELL 7 — Deployment Decision + Dissertation Figures
# ================================================================
# Deployment rule:
#   1. Select retriever with highest LOO-CV accuracy.
#   2. Tie-break: prefer simpler (raw > latent; weighted > unweighted).
#   3. Record latency — if winner is >10× slower than sklearn, note it.
#
# Figures:
#   A. LOO-CV accuracy comparison (bar chart)
#   B. Retrieval latency comparison (bar chart)
#   C. t-SNE: raw feature space vs DAE latent space (side-by-side)
# ================================================================

# ── Deployment decision ──────────────────────────────────────
# Build ordered list by LOO accuracy (exclude reference-only entries)
ranked = sorted(
    [(name, r) for name, r in benchmark_results.items()
     if r.get('mean_latency_ms') is not None],
    key=lambda x: x[1]['loo_accuracy'],
    reverse=True
)
winner_name, winner_stats = ranked[0]

# Determine which FAISS index to load at inference
INDEX_FILE_MAP = {
    'faiss_raw_unweighted': 'faiss_rawspace_unweighted.index',
    'faiss_raw_weighted':   'faiss_rawspace.index',
    'faiss_latent_l2':      'faiss_latent_l2.index',
    'faiss_latent_ip':      'faiss_latent_ip.index',
    'sklearn_cosine':       None,   # no FAISS file — use sklearn
}

deployment = {
    'deployed_retriever':    winner_name,
    'deployed_index_file':   INDEX_FILE_MAP.get(winner_name, 'faiss_rawspace.index'),
    'loo_cv_accuracy':       winner_stats['loo_accuracy'],
    'mean_latency_ms':       winner_stats['mean_latency_ms'],
    'k_neighbours':          K_NEIGHBOURS,
    'uses_shap_weights':     'weighted' in winner_name,
    'uses_dae_latent':       'latent' in winner_name,
    'all_results':           benchmark_results,
    'selection_rationale':   (
        f'{winner_name} achieved the highest LOO-CV accuracy ({winner_stats["loo_accuracy"]:.4f}) '
        f'at {winner_stats["mean_latency_ms"]:.3f} ms/query mean latency. '
        f'Occam\'s razor: simpler model preferred on equal accuracy.'
    )
}

with open(ARTIFACT_DIR / 'vectordb_deployment.json', 'w') as f:
    json.dump(deployment, f, indent=2)

print('=' * 60)
print('DEPLOYMENT DECISION')
print('=' * 60)
print(f'  Winner           : {winner_name}')
print(f'  LOO-CV accuracy  : {winner_stats["loo_accuracy"]:.4f}')
print(f'  Mean latency     : {winner_stats["mean_latency_ms"]:.3f} ms/query')
print(f'  Index file       : {deployment["deployed_index_file"]}')
print()
print('Full ranking:')
for i, (name, r) in enumerate(ranked):
    marker = ' ← DEPLOYED' if i == 0 else ''
    print(f'  {i+1}. {name:<35} LOO={r["loo_accuracy"]:.4f}{marker}')

# ── Figure A: LOO-CV accuracy comparison ─────────────────────
names_plot = [n for n, _ in ranked]
accs_plot  = [r['loo_accuracy'] for _, r in ranked]
lats_plot  = [r['mean_latency_ms'] for _, r in ranked]

# Add Phase 2A Gower reference line if available
ref_line_val = benchmark_results.get('phase2a_gower_reported', {}).get('loo_accuracy')

fig, ax = plt.subplots(figsize=(10, 4.5))
bar_colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(names_plot))]
bars = ax.bar(range(len(names_plot)), accs_plot, color=bar_colors, alpha=0.85,
              edgecolor='white', width=0.6)
for bar, val in zip(bars, accs_plot):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
if ref_line_val:
    ax.axhline(ref_line_val, color='#2c3e50', ls='--', lw=1.5,
               label=f'Phase 2A Gower LOO={ref_line_val:.3f}')
    ax.legend(fontsize=9)
ax.set_xticks(range(len(names_plot)))
ax.set_xticklabels([n.replace('_', '\n') for n in names_plot], fontsize=9)
ax.set_ylabel('LOO-CV Accuracy (k=5 majority vote)', fontsize=11)
ax.set_ylim(max(0, min(accs_plot) - 0.05), 1.02)
ax.set_title('Vector Database Retriever LOO-CV Benchmark\n'
             'C3 Intervention Retrieval Accuracy (n=1,073 cases)', fontsize=12)
deployed_patch = mpatches.Patch(color='#e74c3c', alpha=0.85, label='Deployed')
other_patch    = mpatches.Patch(color='#3498db', alpha=0.85, label='Compared')
ax.legend(handles=[deployed_patch, other_patch], fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'vectordb_loo_accuracy.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: vectordb_loo_accuracy.png ✓')

# ── Figure B: Latency comparison ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3.5))
bars2 = ax.bar(range(len(names_plot)), lats_plot, color='#9b59b6',
               alpha=0.82, edgecolor='white', width=0.6)
for bar, val in zip(bars2, lats_plot):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f'{val:.3f}ms', ha='center', fontsize=9)
ax.set_xticks(range(len(names_plot)))
ax.set_xticklabels([n.replace('_', '\n') for n in names_plot], fontsize=9)
ax.set_ylabel('Mean Query Latency (ms)', fontsize=11)
ax.set_title('Retriever Latency Comparison (single query, CPU, n=1,073)', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'vectordb_latency.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: vectordb_latency.png ✓')

# ── Figure C: t-SNE raw vs latent (if DAE available) ─────────
if DAE_AVAILABLE and Z_cases is not None:
    print('Computing t-SNE (this takes ~30s)...')
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=500)

    # Subsample for speed
    N_TSNE = min(500, len(X_cases))
    idx_s  = np.random.choice(len(X_cases), N_TSNE, replace=False)

    Z2_raw    = tsne.fit_transform(X_cases[idx_s])
    tsne2     = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=500)
    Z2_latent = tsne2.fit_transform(Z_cases[idx_s])
    labels_s  = y_risk[idx_s]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax_i, (Z2, title) in enumerate([
        (Z2_raw,    'Raw Feature Space (13-dim → t-SNE)'),
        (Z2_latent, 'DAE Latent Space (8-dim → t-SNE)')
    ]):
        ax = axes[ax_i]
        for cls, label in [(0,'Low'), (1,'Medium'), (2,'High')]:
            mask = labels_s == cls
            ax.scatter(Z2[mask, 0], Z2[mask, 1],
                       c=PALETTE[label], s=18, alpha=0.65,
                       label=f'{label} (n={mask.sum()})', edgecolors='none')
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=9, markerscale=1.8)
        ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
        ax.spines[['top', 'right']].set_visible(False)
    plt.suptitle(f'Risk Tier Separability: Raw vs DAE Latent Space\n'
                 f'(n={N_TSNE} cases, coloured by true risk tier)', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / 'vectordb_tsne.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: vectordb_tsne.png ✓')

print('Cell 7 complete ✓')

In [ ]:
# ================================================================
# CELL 8 — FINAL SAVE, VERIFY + DOWNLOAD  (v3-fixed)
# ================================================================
import os, shutil, json
from pathlib import Path

ARTIFACT_DIR = Path('c3_vectordb_artifacts')
FIGURE_DIR   = Path('c3_vectordb_figures')
ARTIFACT_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------------
# Always export a canonical seed_case_base.csv from df_cases
# (whether df_cases came from a real zip entry or the surrogate builder)
# ----------------------------------------------------------------
if 'df_cases' in globals():
    canonical_path = ARTIFACT_DIR / 'seed_case_base.csv'
    df_cases.to_csv(canonical_path, index=False)
    print(f'  ✓ seed_case_base.csv  ({len(df_cases)} rows)  →  canonical copy')

    # Remove the old "surrogate" file if it still exists, so the zip
    # only contains the canonical filename
    surrogate = ARTIFACT_DIR / 'seed_case_base_surrogate.csv'
    if surrogate.exists():
        surrogate.unlink()
        print('    (removed old seed_case_base_surrogate.csv)')
else:
    print('  ✗ df_cases not in globals — seed_case_base.csv cannot be written')

# ----------------------------------------------------------------
# Copy faiss_metadata.json into place if named differently
# (some versions wrote it to ARTIFACT_DIR already)
# ----------------------------------------------------------------
meta_targets = list(ARTIFACT_DIR.glob('*metadata*.json'))
target = ARTIFACT_DIR / 'faiss_metadata.json'
if not target.exists() and meta_targets:
    shutil.copy2(meta_targets[0], target)
    print(f'  ✓ faiss_metadata.json   (copied from {meta_targets[0].name})')

# ----------------------------------------------------------------
# Checklist
# ----------------------------------------------------------------
EXPECTED = [
    ('faiss_rawspace.index',             'FAISS raw-space SHAP-weighted cosine (PRIMARY)'),
    ('faiss_rawspace_unweighted.index',  'FAISS raw-space unweighted cosine (ablation)'),
    ('faiss_metadata.json',              'Case metadata for FAISS'),
    ('seed_case_base.csv',               'Canonical 1,073-case seed base'),
    ('vectordb_benchmark.json',          'LOO-CV results for all retrievers'),
    ('vectordb_deployment.json',         'Winning retriever config'),
]
OPTIONAL = [
    ('faiss_latent_l2.index',            'DAE latent L2 (if DAE loaded)'),
    ('faiss_latent_ip.index',            'DAE latent IP (if DAE loaded)'),
]

print('\n' + '=' * 65)
print('PHASE 2C ARTIFACT CHECKLIST')
print('=' * 65)
missing = []
for fname, desc in EXPECTED:
    path = ARTIFACT_DIR / fname
    if path.exists():
        sz = path.stat().st_size / 1024
        print(f'  ✓ {fname:<40} ({sz:>8.1f} KB)  {desc}')
    else:
        print(f'  ✗ {fname:<40} MISSING          {desc}')
        missing.append(fname)

print('\nOptional:')
for fname, desc in OPTIONAL:
    path = ARTIFACT_DIR / fname
    tag = '✓' if path.exists() else '─'
    print(f'  {tag} {fname:<40}                {desc}')

# ChromaDB directory
chroma_dir = Path('chromadb_store')
if chroma_dir.exists() and any(chroma_dir.iterdir()):
    target_chroma = ARTIFACT_DIR / 'chromadb_store'
    if target_chroma.exists():
        shutil.rmtree(target_chroma)
    shutil.copytree(chroma_dir, target_chroma)
    print(f'  ✓ chromadb_store/    (ChromaDB persistent store)')

# Copy figures
fig_target = ARTIFACT_DIR / 'figures'
if FIGURE_DIR.exists():
    shutil.copytree(str(FIGURE_DIR), str(fig_target), dirs_exist_ok=True)

# ----------------------------------------------------------------
# Zip
# ----------------------------------------------------------------
zip_path = '/content/C3_VectorDB_Artifacts.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive('/content/C3_VectorDB_Artifacts', 'zip', str(ARTIFACT_DIR))
zsize = os.path.getsize(zip_path) / (1024 * 1024)

print()
print('=' * 65)
print(f'ZIPPED → C3_VectorDB_Artifacts.zip  ({zsize:.2f} MB)')
print('=' * 65)

if missing:
    print()
    print('⚠ Missing artifacts — Phase 3 will use fallbacks:')
    for m in missing:
        print(f'    - {m}')
    print('  (Phase 3 has a euclidean fallback when FAISS index is missing.)')

# ----------------------------------------------------------------
# Download
# ----------------------------------------------------------------
from google.colab import files
files.download(zip_path)

print()
print('=' * 65)
print('PHASE 2C COMPLETE')
print('=' * 65)
print("""
WHAT YOU NOW HAVE:
  - FAISS SHAP-weighted cosine retriever (primary deployment)
  - ChromaDB metadata-filtered store (intervention + risk_tier filtering)
  - LOO-CV benchmark vs sklearn baseline
  - seed_case_base.csv with canonical filename

NEXT → Phase 3: upload the following into c3_api/artifacts/:
  From C3_FIXED_ARTIFACTS.zip:
    xgboost_smotenc.pkl, probability_calibrator.pkl,
    conformal_predictor.pkl, seed_case_base.csv,
    shap_explainer.pkl, feature_cols.json, ...
  From C3_VectorDB_Artifacts.zip (this one):
    faiss_rawspace.index, faiss_metadata.json
""")
